# exp073 — validación externa zero-shot de exp045 en MRNet

Padre: exp045_raptor_aug_espacial (checkpoint all-data SWA 13–15). Única variable: conjunto evaluado.
No entrena ni usa el split, pseudo-etiquetas o gold de RSNA. Convierte MRNet al corpus Raptor 96×384 y evalúa
anormalidad, LCA y menisco con y sin TTA espejo anatómico.

SPEC: `SPEC.md` en esta carpeta. Celdas fijas: **0** parámetros · **1** setup · **2** datos externos ·
**3** modelo · **4** inferencia · **5** evaluación · **6** guardar · **7** verdict.


## 0. Parámetros (única celda con valores)


In [ ]:
EXP = "exp073_mrnet_val_externa"
PADRE = "exp045"
AUTHOR = "rodrigo"
SMOKE = True
SEED = 42

MRNET_DIR = "/workspace/data/external/mrnet/MRNet-v1.0"
MRNET_LOCAL = "/root/mrnet96_384"
MODELS = {
    "exp045_alldata": "/workspace/users/rodrigo/wt_exp045_raptor_aug_espacial/results/exp045_raptor_aug_espacial/swa_alldata.pt",
}
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
IMG = 384
RES = 320
NORM = "imagenet"
K_EVAL = 24
TTA_MIRROR = True
N_BOOT = 2000
BATCH = 4
N_WORKERS = 8
BUILD_PROCS = 8
FLIP_FIX = {"sagittal": None, "coronal": None, "axial": None}
NOTES = "Validación externa zero-shot de exp045 all-data en MRNet; sin entrenamiento, split RSNA ni gold"


## 1. Setup


In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from multiprocessing import Pool
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS
from src.knee.experiments import git_info, guardar_metricas, seed_everything

assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe y confirma la SPEC"
assert set(FLIP_FIX) == {"sagittal", "coronal", "axial"}
assert all(v in (None, "ud", "lr", "rot90", "rot180", "rot270") for v in FLIP_FIX.values())
assert all(name.replace("_", "").replace("-", "").isalnum() for name in MODELS), "Nombre de modelo no apto para filename"

RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · "
      f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'} · "
      f"timm {timm.__version__} · torch {torch.__version__}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")


## 2. Datos externos: MRNet → corpus Raptor 96×384


In [ ]:
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
SLOT_SIZES = [s[2] for s in SLOTS]
SLOT_STARTS = np.cumsum([0] + SLOT_SIZES[:-1]).tolist()
EMPTY_SLOTS = [1, 2]
SPAN = (0.02, 0.98)
D = sum(SLOT_SIZES)
assert D == 96


def _apply_flip(array, fix):
    """Aplica una corrección de orientación a (S,H,W), sin cambiar código."""
    if fix is None:
        out = array
    elif fix == "ud":
        out = np.flip(array, axis=1)
    elif fix == "lr":
        out = np.flip(array, axis=2)
    elif fix == "rot90":
        out = np.rot90(array, 1, axes=(1, 2))
    elif fix == "rot180":
        out = np.rot90(array, 2, axes=(1, 2))
    elif fix == "rot270":
        out = np.rot90(array, 3, axes=(1, 2))
    else:
        raise ValueError(f"FLIP_FIX desconocido: {fix}")
    return np.ascontiguousarray(out)


def _series_to_slot(series, k, fix, img=384):
    """Muestrea 2–98 %, normaliza p2/p98 por serie y redimensiona los cortes elegidos."""
    series = np.asarray(series)
    if series.ndim != 3 or len(series) == 0:
        raise ValueError(f"Serie MRNet inválida: shape={series.shape}")
    n = len(series)
    lo = int(n * 0.02)
    hi = max(int(n * 0.98) - 1, lo)
    picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else np.zeros(k, dtype=int)
    picked = _apply_flip(series[np.clip(picks, 0, n - 1)], fix).astype(np.float32)
    p2, p98 = np.percentile(picked, [2.0, 98.0])
    scaled = np.clip((picked - p2) / (p98 - p2 + 1e-6), 0.0, 1.0)
    interpolation = cv2.INTER_AREA if img < min(series.shape[1:]) else cv2.INTER_LINEAR
    resized = [cv2.resize(a, (img, img), interpolation=interpolation) for a in scaled]
    return (np.stack(resized) * 255).astype(np.uint8)


def build_mrnet_volume(series_by_plane, flip_fix, img=384):
    """Mapea MRNet a slots Raptor: sagital→0, coronal→3, axial→4; 1 y 2 quedan vacíos."""
    slot_sizes = [26, 22, 18, 12, 18]
    slot_starts = np.cumsum([0] + slot_sizes[:-1]).tolist()
    plane_to_slot = {"sagittal": 0, "coronal": 3, "axial": 4}
    vol = np.zeros((sum(slot_sizes), img, img), dtype=np.uint8)
    mask = np.zeros(sum(slot_sizes), dtype=np.uint8)
    for plane, slot in plane_to_slot.items():
        k = slot_sizes[slot]
        start = slot_starts[slot]
        vol[start:start + k] = _series_to_slot(series_by_plane[plane], k, flip_fix[plane], img=img)
        mask[start:start + k] = 1
    return vol, mask


def _read_mrnet_labels(root):
    frames = []
    for split, expected in (("train", 1130), ("valid", 120)):
        merged = None
        for task in ("abnormal", "acl", "meniscus"):
            path = Path(root) / f"{split}-{task}.csv"
            part = pd.read_csv(path, header=None, names=["case", task])
            part["case"] = part["case"].astype(int).map(lambda x: f"{x:04d}")
            merged = part if merged is None else merged.merge(part, on="case", validate="one_to_one")
        assert len(merged) == expected, f"{split}: {len(merged)} != {expected}"
        assert merged[["abnormal", "acl", "meniscus"]].isin([0, 1]).all().all()
        merged.insert(1, "split", split)
        frames.append(merged)
    out = pd.concat(frames, ignore_index=True)
    out["dataset_id"] = out["split"] + "/" + out["case"]
    assert not out["dataset_id"].duplicated().any()
    return out


_MRNET_WORKER = {}


def _init_mrnet_worker(out_dir, mrnet_root):
    _MRNET_WORKER["vols"] = np.load(Path(out_dir) / "vols.npy", mmap_mode="r+")
    _MRNET_WORKER["root"] = Path(mrnet_root)


def _build_mrnet_exam(job):
    row, split, case = job
    started = time.time()
    try:
        series = {
            plane: np.load(_MRNET_WORKER["root"] / split / plane / f"{case}.npy", mmap_mode="r")
            for plane in ("sagittal", "coronal", "axial")
        }
        vol, mask = build_mrnet_volume(series, FLIP_FIX, img=IMG)
        error = None
    except Exception as exc:
        vol = np.zeros((D, IMG, IMG), dtype=np.uint8)
        mask = np.zeros(D, dtype=np.uint8)
        error = f"{type(exc).__name__}: {exc}"
    _MRNET_WORKER["vols"][row] = vol
    return row, mask, error, time.time() - started


def _build_mrnet_corpus(labels):
    out = Path(MRNET_LOCAL)
    out.mkdir(parents=True, exist_ok=True)
    ids = labels["dataset_id"].to_numpy(dtype=str)
    vols = np.lib.format.open_memmap(out / "vols.npy", mode="w+", dtype=np.uint8,
                                     shape=(len(ids), D, IMG, IMG))
    del vols
    np.save(out / "ids.npy", ids)
    masks = np.zeros((len(ids), D), dtype=np.uint8)
    errors = {}
    jobs = [(row, rec.split, rec.case) for row, rec in enumerate(labels.itertuples(index=False))]
    started = time.time()
    with Pool(BUILD_PROCS, initializer=_init_mrnet_worker, initargs=(out, MRNET_DIR)) as pool:
        for done, (row, mask, error, seconds) in enumerate(
                pool.imap_unordered(_build_mrnet_exam, jobs, chunksize=1), start=1):
            masks[row] = mask
            if error:
                errors[ids[row]] = error
                print(f"[error] {ids[row]}: {error}", flush=True)
            if done % 50 == 0 or done == len(ids):
                elapsed = time.time() - started
                rate = done / max(elapsed, 1e-6)
                print(f"{done}/{len(ids)} · {elapsed:.0f}s · {rate:.2f} estudios/s · "
                      f"ETA {(len(ids) - done) / max(rate, 1e-6) / 60:.1f} min · último {seconds:.1f}s",
                      flush=True)
    np.save(out / "masks.npy", masks)
    meta = {
        "n": len(ids), "seconds": time.time() - started, "img": IMG, "D": D,
        "slots": SLOTS, "span": SPAN, "empty_slots": EMPTY_SLOTS, "flip_fix": FLIP_FIX,
        "procs": BUILD_PROCS, "errors": errors,
        "valid_slices_mean": float(masks.sum(1).mean()),
        "empty_studies": int((masks.sum(1) == 0).sum()),
        "source": "MRNet-v1.0; 256 px field assumed approximately equal to RSNA 140 mm crop",
    }
    (out / "build_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Corpus MRNet construido en {meta['seconds']:.1f}s · errores {len(errors)}")
    return meta


mrnet_labels = _read_mrnet_labels(MRNET_DIR)
_corpus_files = [Path(MRNET_LOCAL) / name for name in ("vols.npy", "masks.npy", "ids.npy", "build_meta.json")]
if not all(path.exists() for path in _corpus_files):
    corpus_meta = _build_mrnet_corpus(mrnet_labels)
else:
    corpus_meta = json.loads((Path(MRNET_LOCAL) / "build_meta.json").read_text(encoding="utf-8"))
    assert corpus_meta.get("flip_fix") == FLIP_FIX, (
        "El corpus cacheado usa otro FLIP_FIX; elimina MRNET_LOCAL para reconstruirlo con el parámetro actual"
    )
    print(f"Corpus MRNet cacheado · build {corpus_meta.get('seconds', float('nan')):.1f}s")

corpus_ids = np.load(Path(MRNET_LOCAL) / "ids.npy").astype(str)
corpus_masks = np.load(Path(MRNET_LOCAL) / "masks.npy", mmap_mode="r")
ID2ROW = {uid: row for row, uid in enumerate(corpus_ids)}
assert len(corpus_ids) == 1250 and set(corpus_ids) == set(mrnet_labels["dataset_id"])
assert corpus_masks.shape == (1250, D)
assert not corpus_meta.get("errors"), f"Fallaron {len(corpus_meta['errors'])} estudios al construir el corpus"

if SMOKE:
    mrnet_eval = pd.concat(
        [mrnet_labels.loc[mrnet_labels["split"] == split].head(16) for split in ("train", "valid")],
        ignore_index=True,
    )
else:
    mrnet_eval = mrnet_labels.copy()
eval_ids = mrnet_eval["dataset_id"].tolist()
print(f"MRNet: corpus {len(corpus_ids)} · inferencia {len(eval_ids)} "
      f"({mrnet_eval.groupby('split').size().to_dict()}) · positivos "
      f"{mrnet_eval[['abnormal', 'acl', 'meniscus']].sum().astype(int).to_dict()}")

# Comprobación visual: MRNet y RSNA, tres ejemplos por plano. MRNet no incluye PixelSpacing;
# se asume que su campo 256×256 es aproximadamente el recorte físico RSNA de 140 mm.
rsna_path = next((p for p in (Path("/root/corpus96_384/vols.npy"),
                              Path("/workspace/data/cache/raptor96_384/vols.npy")) if p.exists()), None)
assert rsna_path is not None, "No se encontró un corpus RSNA para orientation_check.png"
mrnet_vols = np.load(Path(MRNET_LOCAL) / "vols.npy", mmap_mode="r")
rsna_vols = np.load(rsna_path, mmap_mode="r")
checks = [("Sagittal", 0), ("Coronal", 3), ("Axial", 4)]
fig, axes = plt.subplots(len(checks), 6, figsize=(15, 7))
for row, (plane, slot) in enumerate(checks):
    center = SLOT_STARTS[slot] + SLOT_SIZES[slot] // 2
    for col in range(3):
        axes[row, col].imshow(mrnet_vols[ID2ROW[eval_ids[col]], center], cmap="gray")
        axes[row, col].set_title(f"MRNet {mrnet_eval.iloc[col]['case']}" if row == 0 else "")
        axes[row, col + 3].imshow(rsna_vols[col, center], cmap="gray")
        axes[row, col + 3].set_title(f"RSNA row {col}" if row == 0 else "")
    axes[row, 0].set_ylabel(plane)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"Orientation check · FLIP_FIX={FLIP_FIX}")
fig.tight_layout()
orientation_path = RESULTS / "orientation_check.png"
fig.savefig(orientation_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print("Guardado:", orientation_path)


## 3. Modelo (Raptor/exp045, solo ruta de evaluación)


In [ ]:
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
SAGITTAL_SLOTS = (0, 1)
SLOT_OF = np.repeat(np.arange(len(SLOT_SIZES)), SLOT_SIZES)
assert len(SLOT_OF) == D


class StudyWindows(Dataset):
    """Bolsa eval de K ventanas 2.5D; centros linspace sobre el rango válido, como exp045."""
    def __init__(self, ids, k, train=False):
        self.ids, self.k, self.train = list(ids), k, train
        self.vols = None

    def __len__(self):
        return len(self.ids)

    def _centers(self, valid, count):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, D - 2))]
        assert not self.train, "exp073 solo admite el camino eval (train=False)"
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return [cs[i] for i in idx]

    def __getitem__(self, i):
        if self.vols is None:
            self.vols = np.load(Path(MRNET_LOCAL) / "vols.npy", mmap_mode="r")
        uid = self.ids[i]
        row = ID2ROW[uid]
        valid = np.where(corpus_masks[row] > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, D))
        cs = np.array([max(1, min(c, D - 2)) for c in self._centers(valid, self.k)])
        lo = (IMG - RES) // 2
        vol = self.vols[row]
        tri = np.stack(
            [vol[cs - 1, lo:lo + RES, lo:lo + RES],
             vol[cs, lo:lo + RES, lo:lo + RES],
             vol[cs + 1, lo:lo + RES, lo:lo + RES]],
            axis=1,
        )
        sag = torch.from_numpy(np.isin(SLOT_OF[cs], SAGITTAL_SLOTS))
        return (torch.from_numpy(np.ascontiguousarray(tri)),
                torch.tensor(1.0, dtype=torch.float32), i, sag)


def to_input(x_u8, gain):
    x = x_u8.to(DEVICE, non_blocking=True).float() / 255.0
    if not torch.all(gain == 1.0):
        x = (x * gain.to(DEVICE).view(-1, 1, 1, 1, 1)).clamp(0, 1)
    if NORM == "imagenet":
        x = (x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)
    return x


def build_backbone(arch, pretrained=True):
    return timm.create_model(arch, pretrained=pretrained, num_classes=0, in_chans=3,
                             global_pool="avg", img_size=RES)


class RaptorClassifier(nn.Module):
    """Raptor encoder + per-diagnosis attention-MIL head (12 hallazgos)."""
    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K_ = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).view(B, K_, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.encode(x))


def build_model(arch):
    backbone = build_backbone(arch, pretrained=False)
    return RaptorClassifier(backbone, F_dim=backbone.num_features).to(DEVICE)


def _mirror_tri(x, sag):
    """TTA nartaa: sagital invierte los 3 canales; coronal/axial voltea el ancho."""
    sag = sag.view(x.shape[0], x.shape[1], 1, 1, 1)
    return torch.where(sag, x.flip(2), x.flip(-1))


## 4. Inferencia por checkpoint (sin entrenamiento)


In [ ]:
def inference_loader(ids):
    return DataLoader(
        StudyWindows(ids, K_EVAL, train=False),
        batch_size=BATCH,
        shuffle=False,
        drop_last=False,
        num_workers=N_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=False,
    )


@torch.no_grad()
def predict_checkpoint(model, ids):
    model.eval()
    pred_no_tta = np.zeros((len(ids), len(LABELS)), dtype=np.float32)
    pred_tta = np.zeros_like(pred_no_tta)
    for x, gain, idx, sag in inference_loader(ids):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            p0 = torch.sigmoid(model(to_input(x, gain)).float())
            if TTA_MIRROR:
                p1 = torch.sigmoid(model(to_input(_mirror_tri(x, sag), gain)).float())
                pt = (p0 + p1) / 2
            else:
                pt = p0
        rows = idx.numpy()
        pred_no_tta[rows] = p0.cpu().numpy()
        pred_tta[rows] = pt.cpu().numpy()
    index = pd.Index(ids, name="dataset_id")
    return (pd.DataFrame(pred_no_tta, index=index, columns=LABELS),
            pd.DataFrame(pred_tta, index=index, columns=LABELS))


predictions = {}
inference_info = {}
for model_name, checkpoint_path in MODELS.items():
    started = time.time()
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    missing_keys = {"model", "arch", "res", "lab"} - set(checkpoint)
    assert not missing_keys, f"{model_name}: checkpoint sin claves {sorted(missing_keys)}"
    assert checkpoint["arch"] == ARCH, (checkpoint["arch"], ARCH)
    assert int(checkpoint["res"]) == RES, (checkpoint["res"], RES)
    assert list(checkpoint["lab"]) == list(LABELS), "Orden de LABELS distinto al checkpoint"
    model = build_model(ARCH)
    model.load_state_dict(checkpoint["model"], strict=True)
    pred_no_tta, pred_tta = predict_checkpoint(model, eval_ids)
    predictions[model_name] = {"no_tta": pred_no_tta, "tta": pred_tta}

    export = mrnet_eval[["case", "split", "dataset_id"]].set_index("dataset_id").copy()
    export = export.join(pred_no_tta.add_prefix("no_tta__")).join(pred_tta.add_prefix("tta__"))
    pred_path = RESULTS / f"preds_{model_name}.csv"
    export.reset_index(drop=True).to_csv(pred_path, index=False)
    elapsed = time.time() - started
    inference_info[model_name] = {
        "checkpoint": checkpoint_path,
        "predictions": str(pred_path),
        "seconds": elapsed,
        "n_exams": len(eval_ids),
        "tta_mirror": TTA_MIRROR,
    }
    print(f"{model_name}: {len(eval_ids)} estudios · {elapsed:.1f}s · {pred_path}")
    del model, checkpoint
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


## 5. Evaluación MRNet (AUC + bootstrap 95 %)


In [ ]:
def bootstrap_auc(y_true, y_score, n_boot=2000, seed=42):
    """AUC e IC percentil por remuestreo de exámenes; omite réplicas con una sola clase."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)
    keep = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = y_true[keep].astype(int)
    y_score = y_score[keep]
    n = len(y_true)
    n_pos = int(y_true.sum())
    if n == 0 or len(np.unique(y_true)) < 2:
        return {"auc": None, "ci_lo": None, "ci_hi": None, "n": n, "n_pos": n_pos}
    auc = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(n_boot):
        sample = rng.integers(0, n, size=n)
        if len(np.unique(y_true[sample])) == 2:
            boot.append(roc_auc_score(y_true[sample], y_score[sample]))
    if not boot:
        return {"auc": auc, "ci_lo": None, "ci_hi": None, "n": n, "n_pos": n_pos}
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    return {"auc": auc, "ci_lo": float(ci_lo), "ci_hi": float(ci_hi), "n": n, "n_pos": n_pos}


def _mrnet_task_scores(pred):
    values = pred[LABELS].to_numpy(dtype=float)
    return {
        "acl": pred["ACL"].to_numpy(dtype=float),
        "meniscus": pred[["Medial Meniscus", "Lateral Meniscus"]].max(axis=1).to_numpy(dtype=float),
        "abnormal_max": values.max(axis=1),
        "abnormal_noisy_or": 1.0 - np.prod(1.0 - values, axis=1),
    }


labels_by_id = mrnet_labels.set_index("dataset_id")
mrnet_metrics = {}
auc_rows = []
for model_name, variants in predictions.items():
    mrnet_metrics[model_name] = {}
    for split_name in ("valid", "train", "all"):
        ids = (mrnet_eval["dataset_id"].tolist() if split_name == "all" else
               mrnet_eval.loc[mrnet_eval["split"] == split_name, "dataset_id"].tolist())
        mrnet_metrics[model_name][split_name] = {}
        truth = labels_by_id.loc[ids]
        for variant_name, pred_all in variants.items():
            pred = pred_all.loc[ids]
            for task_name, score in _mrnet_task_scores(pred).items():
                label_name = "abnormal" if task_name.startswith("abnormal") else task_name
                result = bootstrap_auc(truth[label_name].to_numpy(), score, N_BOOT, SEED)
                metric_name = f"{task_name}_{variant_name}"
                mrnet_metrics[model_name][split_name][metric_name] = result
                auc_rows.append({"model": model_name, "split": split_name, "metric": metric_name, **result})

auc_table = pd.DataFrame(auc_rows)
display(auc_table.assign(
    auc=auc_table["auc"].round(4),
    ci=auc_table.apply(
        lambda r: "—" if pd.isna(r["ci_lo"]) else f"[{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]", axis=1
    ),
)[["model", "split", "metric", "auc", "ci", "n", "n_pos"]])

# Lectura informativa: cada salida RSNA contra cada etiqueta MRNet, por conjunto.
per_label_rows = []
for model_name, variants in predictions.items():
    for variant_name, pred_all in variants.items():
        for split_name in ("valid", "train", "all"):
            ids = (mrnet_eval["dataset_id"].tolist() if split_name == "all" else
                   mrnet_eval.loc[mrnet_eval["split"] == split_name, "dataset_id"].tolist())
            pred = pred_all.loc[ids]
            truth = labels_by_id.loc[ids]
            for target in ("abnormal", "acl", "meniscus"):
                y = truth[target].to_numpy()
                for label in LABELS:
                    auc = float(roc_auc_score(y, pred[label])) if len(np.unique(y)) == 2 else None
                    per_label_rows.append({
                        "model": model_name, "variant": variant_name, "split": split_name,
                        "target": target, "rsna_label": label, "auc": auc,
                    })
per_label_auc = pd.DataFrame(per_label_rows)
for (model_name, variant_name, split_name), block in per_label_auc.groupby(
        ["model", "variant", "split"], sort=False):
    print(f"\nAUC por hallazgo RSNA vs etiqueta MRNet · {split_name} · {model_name} · {variant_name}")
    display(block.pivot(index="rsna_label", columns="target", values="auc").round(4))

print("\nMRNet paper (test privado; contexto, no comparación 1:1): "
      "abnormal 0.937 · ACL 0.965 · meniscus 0.847")

DECISIONS = {}
for model_name in MODELS:
    if SMOKE:
        DECISIONS[model_name] = "smoke"
        continue
    valid = mrnet_metrics[model_name]["valid"]
    abnormal = valid["abnormal_max_tta"]["auc"]
    acl = valid["acl_tta"]["auc"]
    meniscus = valid["meniscus_tta"]["auc"]
    DECISIONS[model_name] = ("killed" if acl < 0.60 else
                             "worked" if abnormal >= 0.85 and acl >= 0.85 and meniscus >= 0.75
                             else "didnt")
print("Decisiones automáticas:", DECISIONS)


## 6. Guardar `results/<EXP>/metrics.json`


In [ ]:
PARAMS = {
    "SMOKE": SMOKE, "SEED": SEED, "MRNET_DIR": MRNET_DIR, "MRNET_LOCAL": MRNET_LOCAL,
    "MODELS": MODELS, "ARCH": ARCH, "IMG": IMG, "RES": RES, "NORM": NORM,
    "K_EVAL": K_EVAL, "TTA_MIRROR": TTA_MIRROR, "N_BOOT": N_BOOT, "BATCH": BATCH,
    "N_WORKERS": N_WORKERS, "BUILD_PROCS": BUILD_PROCS, "FLIP_FIX": FLIP_FIX, "NOTES": NOTES,
}
metrics = {
    "schema_version": 1,
    "exp": EXP,
    "author": AUTHOR,
    "parent": PADRE,
    "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else "external_val",
    "params": PARAMS,
    "mrnet": mrnet_metrics,
    "corpus": {
        "path": MRNET_LOCAL,
        "n_exams": len(corpus_ids),
        "seconds": corpus_meta.get("seconds"),
        "shape": [len(corpus_ids), D, IMG, IMG],
        "empty_slots": EMPTY_SLOTS,
        "empty_slot_names": ["Sagittal non-fluid", "Coronal fluid"],
        "flip_fix": FLIP_FIX,
        "n_errors": len(corpus_meta.get("errors", {})),
        "orientation_check": str(orientation_path),
    },
    "inference": inference_info,
    "gpu_hours_total": (
        sum(info["seconds"] for info in inference_info.values()) / 3600 if DEVICE.type == "cuda" else 0.0
    ),
    "decision_auto": DECISIONS,
}
print("Guardado:", guardar_metricas(RESULTS, metrics))


## 7. Verdict

Revisa primero `results/<EXP>/orientation_check.png`. Después de la corrida real, copia los AUC e IC de
`metrics.json` a la sección 3 de `SPEC.md` y actualiza `docs/EXPERIMENTOS.md` con `worked`, `didnt` o `killed`.
No uses MRNet para ajustar umbrales, calibrar ni seleccionar checkpoints.
